# Phase 02 — Penggabungan Bank, Pemecahan Kalimat & EDA

Notebook **mandiri (self-contained)**. Mengubah data mentah artikel (hasil Phase 01) menjadi data **per kalimat** untuk BERTopic (Phase 03), lalu melakukan **EDA** sebagai **analisis saja**.

### Prinsip
- **Dateline** ('Kota, CNBC Indonesia -') sudah dibuang di Phase 01, jadi tidak ditangani lagi di sini.
- **Bagian EDA bersifat ANALISIS, bukan transformasi.** Tidak ada perubahan permanen (casefolding / stopword removal / drop) pada kolom `kalimat` yang disimpan — teks tetap asli & alami agar embedding BERTopic tidak rusak. Casefolding & stopword hanya *ditampilkan* untuk memahami data. Kesimpulan & keputusan dirangkum di sel akhir.
- Satu-satunya pengurangan data: buang kalimat < 5 kata (filter kualitas) + duplikat + kosong (noise sah).
- Kalimat TIDAK difilter berdasarkan kata relevan (aspek harus induktif dari BERTopic).

### Alur
1. Baca `data/raw/raw.csv`
2. Gabung keyword → kolom `bank`
3. Cleaning ringan (footer/email/label — BUKAN casefolding/stopword)
4. Pecah per kalimat (lindungi singkatan `PT.`/`Tbk.`/desimal)
5. **EDA (analisis saja)**: wordcount, panjang kalimat, duplikat, missing, frekuensi kata, stopword, casefolding preview
6. Buang kalimat < 5 kata + duplikat + kosong
7. Kesimpulan
8. Simpan `data/processed/sentences.csv`

## 1. Import & baca data

In [ ]:
import re
from pathlib import Path
from collections import Counter
import pandas as pd

input_csv = "data/raw/raw.csv"          # <- bisa diubah: sumber data mentah (hasil Phase 01)
df = pd.read_csv(input_csv, dtype={"id_artikel": str})
print(f"Artikel mentah: {len(df)}")
df.head(3)

## 2. Gabung keyword → kolom `bank`

Keyword scraping (BCA, BBCA, BRI, ...) digabung jadi 4 bank. Deteksi dari **judul + isi** (satu artikel sering menyebut beberapa bank). Kolom `bank` berisi daftar bank yang disebut, dipisah koma.

In [ ]:
POLA_BANK = {
    "BCA":     r"\bbca\b|\bbbca\b|bank central asia",          # <- bisa diubah: pola BCA
    "BRI":     r"\bbri\b|\bbbri\b|bank rakyat indonesia|brilink",
    "BNI":     r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}

def deteksi_bank(teks):
    teks = (teks or "").lower()
    return ", ".join(b for b, pat in POLA_BANK.items() if re.search(pat, teks))

df["bank"] = (df["judul"].fillna("") + " " + df["isi_mentah"].fillna("")).map(deteksi_bank)
print("Distribusi jumlah bank yang disebut per artikel:")
print(df["bank"].map(lambda x: len(x.split(", ")) if x else 0).value_counts().sort_index())
print("\nArtikel tanpa bank terdeteksi:", (df["bank"] == "").sum())
df[["id_artikel", "keyword", "bank", "judul"]].head(5)

## 3. Cleaning ringan

Buang noise struktural (footer research, email, label 'Baca:'/'Pilihan Redaksi') + normalisasi spasi. **Tidak** casefolding, **tidak** stopword removal, **tidak** menyaring berdasarkan topik.

In [ ]:
def cleaning_ringan(teks):
    t = str(teks)
    t = re.sub(r"CNBC INDONESIA RESEARCH\s*\S*@\S*", " ", t, flags=re.I)   # footer research
    t = re.sub(r"\S+@\S+\.\S+", " ", t)                                    # email
    t = re.sub(r"Baca\s*:\s*", " ", t, flags=re.I)                         # label 'Baca:'
    t = re.sub(r"Pilihan Redaksi", " ", t, flags=re.I)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["isi_bersih"] = df["isi_mentah"].map(cleaning_ringan)
print("Contoh sesudah cleaning:", df["isi_bersih"].iloc[0][:150])

## 4. Pemecah kalimat (melindungi singkatan Indonesia)

Pemecah naif memotong salah di `PT.`, `Tbk.`, angka desimal (`1.519`). Fungsi ini melindungi titik singkatan dulu, baru memecah di `. ! ?`. Hasil: tabel kalimat dengan `id_artikel`, `id_kalimat`, `bank`, `timestamp`.

In [ ]:
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj",
             "dll", "dsb", "dkk", "a.l", "yakni"]   # <- bisa diubah: tambah singkatan bila perlu

def pecah_kalimat(teks):
    t = re.sub(r"\s+", " ", str(teks)).strip()
    for ab in SINGKATAN:
        t = re.sub(rf"\b{re.escape(ab)}\.", f"{ab}<DOT>", t)
    t = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", t)
    kalimat = re.split(r"(?<=[.!?])\s+", t)
    return [k.replace("<DOT>", ".").strip() for k in kalimat if k.strip()]

baris = []
for _, r in df.iterrows():
    for i, k in enumerate(pecah_kalimat(r["isi_bersih"])):
        baris.append({
            "id_artikel": r["id_artikel"],
            "id_kalimat": f"{r['id_artikel']}_{i}",
            "bank": r["bank"],
            "timestamp": r["timestamp"],
            "kalimat": k,
        })
kal = pd.DataFrame(baris)
kal["n_kata"] = kal["kalimat"].str.split().str.len()
print(f"Total kalimat (sebelum filter apa pun): {len(kal)} dari {len(df)} artikel")
print(f"Rata-rata kalimat/artikel: {len(kal)/len(df):.1f}")

## 5. EDA — ANALISIS SAJA (tidak mengubah data)

Semua sel di bawah hanya **memeriksa** data; tidak ada perubahan permanen. Kesimpulan dirangkum di bagian 7.

### 5a. Distribusi wordcount & panjang kalimat

In [ ]:
print("Statistik jumlah kata per kalimat:")
print(kal["n_kata"].describe().round(1))
print("\nDistribusi panjang kalimat (kata):")
bins = [0, 5, 10, 15, 20, 30, 50, 10000]
print(pd.cut(kal["n_kata"], bins=bins, right=False).value_counts().sort_index())
print("\nStatistik panjang kalimat (karakter):")
print(kal["kalimat"].str.len().describe().round(1))

### 5b. Duplikat & missing (hanya dihitung, belum dibuang)

In [ ]:
n_dup = kal["kalimat"].duplicated().sum()
n_kosong = (kal["kalimat"].str.strip() == "").sum()
n_pendek = (kal["n_kata"] < 5).sum()
print(f"Kalimat duplikat (teks persis sama) : {n_dup} ({n_dup/len(kal)*100:.1f}%)")
print(f"Kalimat kosong/missing              : {n_kosong}")
print(f"Kalimat < 5 kata                    : {n_pendek} ({n_pendek/len(kal)*100:.1f}%)")
print("\nContoh 10 kalimat duplikat yang paling sering muncul:")
dup_counts = kal["kalimat"].value_counts()
for s, c in dup_counts[dup_counts > 1].head(10).items():
    print(f"  ({c}x) {s[:80]}")

### 5c. Inspeksi kalimat < 5 kata (apa yang akan dibuang)

In [ ]:
pendek = kal[kal["n_kata"] < 5]
print(f"Distribusi jumlah kata pada kalimat pendek: {dict(pendek['n_kata'].value_counts().sort_index())}")
print("\n30 contoh kalimat < 5 kata (pastikan memang sampah, bukan relevan):")
for s in pendek["kalimat"].head(30).tolist():
    print("  |", s)

### 5d. Relevansi kata: frekuensi kata (analisis, bukan untuk dibuang)

Catatan: kata ber-frekuensi tinggi di sini **bukan noise untuk dibuang** — sebagian stopword (yang/dan/di) ditangani nanti di vectorizer BERTopic (Phase 03), sebagian lagi kata domain (rp/bank/saham) yang justru inti topik. Tabel ini hanya untuk memahami sebaran kata.

In [ ]:
tokens = []
for k in kal["kalimat"]:
    tokens += re.findall(r"[a-zA-Z]+", str(k).lower())
freq = Counter(tokens)
print(f"Total token: {len(tokens)} | kata unik: {len(freq)}")
print("\n30 kata paling sering:")
for w, c in freq.most_common(30):
    print(f"  {w:18} {c}")

### 5e. Analisis stopword & casefolding (PREVIEW saja — tidak diterapkan ke data)

Menampilkan seberapa besar porsi stopword dan bentuk kalimat bila di-casefold. Ini hanya gambaran; kolom `kalimat` yang disimpan tetap ASLI.

In [ ]:
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    stopwords = set(StopWordRemoverFactory().get_stop_words())
except ImportError:
    print("(Sastrawi belum terpasang: pip install Sastrawi) -- pakai daftar stopword minimal.")
    stopwords = set("yang dan di ke dari untuk pada dengan ini itu atau juga tidak dalam".split())

n_stop = sum(1 for t in tokens if t in stopwords)
print(f"Jumlah stopword Indonesia dikenali: {len(stopwords)}")
print(f"Porsi token stopword: {n_stop}/{len(tokens)} ({n_stop/len(tokens)*100:.1f}%)")

contoh = kal["kalimat"].iloc[0]
print("\n-- PREVIEW transformasi (TIDAK disimpan) --")
print("asli        :", contoh[:90])
print("casefold    :", contoh.lower()[:90])
tanpa_stop = " ".join(w for w in re.findall(r"\w+", contoh.lower()) if w not in stopwords)
print("tanpa stop  :", tanpa_stop[:90])
print("\n(Teks final yang disimpan tetap versi ASLI, bukan hasil preview di atas.)")

## 6. Terapkan filter (kalimat < 5 kata, duplikat, kosong)

Satu-satunya pengurangan data yang dilakukan: buang kalimat terlalu pendek + duplikat + kosong. **Bukan** casefolding/stopword/filter topik.

In [ ]:
AMBANG_KATA = 5   # <- bisa diubah: ambang minimum kata per kalimat

kal_final = kal.copy()
kal_final = kal_final[kal_final["kalimat"].str.strip() != ""]          # buang kosong
kal_final = kal_final[kal_final["n_kata"] >= AMBANG_KATA]              # buang < 5 kata
kal_final = kal_final.drop_duplicates(subset="kalimat").reset_index(drop=True)  # buang duplikat
print(f"Kalimat sebelum filter : {len(kal)}")
print(f"Kalimat setelah filter : {len(kal_final)} (dibuang {len(kal)-len(kal_final)})")

## 7. Kesimpulan EDA

Rangkuman temuan dari analisis di atas (isi/ sesuaikan setelah melihat output di data penuh):

- **Jumlah data**: `N` artikel → `M` kalimat final (rata ± X kalimat/artikel).
- **Panjang kalimat**: mayoritas di rentang ... kata; sangat pendek (<5) dibuang karena sampah struktural (footer, penomoran, pecahan clickbait).
- **Duplikat**: ditemukan `d` kalimat identik (mis. boilerplate berulang) → dibuang.
- **Frekuensi kata**: didominasi stopword + kata domain finansial (rp, bank, saham) — wajar untuk berita bank; tidak ada kata yang dibuang berdasarkan frekuensi.
- **Stopword & casefolding**: hanya dianalisis, TIDAK diterapkan ke teks final (agar embedding BERTopic tetap utuh). Stopword akan ditangani di vectorizer Phase 03.
- **Keputusan**: data per kalimat disimpan apa adanya (teks asli), siap untuk BERTopic.

## 8. Simpan output

In [ ]:
output_csv = Path("data/processed/sentences.csv")   # <- bisa diubah: lokasi output
output_csv.parent.mkdir(parents=True, exist_ok=True)
kolom = ["id_kalimat", "id_artikel", "bank", "timestamp", "kalimat"]
kal_final[kolom].to_csv(output_csv, index=False)
print(f"[SIMPAN] {len(kal_final)} kalimat -> {output_csv}")
kal_final[kolom].head(10)